# Selected-model scib metrics — Blinatumomab (instruction 5)

scib-metrics for the model(s) selected in `final_sweep_benchmark.ipynb` (`cache/final_sweep/selected.json`), compared against the **vanilla MM** baseline (same params, `arch=baseline`) and the two **scVI specialists**.

- **Bio-preservation ×2** — `label_key='cell_type_annot'`, scored on (a) the **joint** latent and (b) the **per-modality** latents (`z_abn`, `z_spt`).
- **System ×1** — `batch_key='cell_system'`, scored on the **joint** latent.
- **Optional** — full `MultiModalVIMetrics` (reconstruction / NLL / latent quality), needs model reload (GPU).

Embeddings are read from cached `latents.npz` — scib needs only the obsm arrays, no model reload.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

PIXELGEN_ROOT = '/home/projects/nyosef/zvise/PixelGen'        # parent -> import as PixelGen.*
PIXELGEN_PKG  = PIXELGEN_ROOT + '/PixelGen'                    # repo dir -> metrics.py's `from scvi_utils import`
for p in (PIXELGEN_ROOT, PIXELGEN_PKG):
    if p not in sys.path:
        sys.path.append(p)

NEW_DATA = Path(PIXELGEN_PKG) / 'New_Data'
CACHE    = NEW_DATA / 'cache'
FINAL    = CACHE / 'final_sweep'
ADATA_PATH = CACHE / 'adata_cytovi_annotated_compat.h5ad'
BIO_KEY, BATCH_KEY = 'cell_type_annot', 'cell_system'

adata = sc.read_h5ad(ADATA_PATH)
bl = np.load(FINAL / 'scvi_baselines.npz')
adata.obsm['z_scvi_abundance'] = bl['z_scvi_abundance']
adata.obsm['z_scvi_spatial']   = bl['z_scvi_spatial']
sel_ids = json.loads((FINAL / 'selected.json').read_text())['selected']
print('selected:', sel_ids)

In [ ]:
# Load cached latents for each selected run + its vanilla-MM counterpart (arch->baseline, same params)
def vanilla_of(rid):
    parts = rid.split('__'); parts[0] = 'arch-baseline'; return '__'.join(parts)

loaded, label = [], {}
def load_run(rid, tag):
    if rid in label:
        return
    p = FINAL / rid / 'latents.npz'
    if not p.exists():
        print(f'  [missing] {rid} ({tag}) — skipped'); return
    z = np.load(p)
    adata.obsm[f'joint::{rid}'] = z['z_joint']
    adata.obsm[f'abn::{rid}']   = z['z_abn']
    adata.obsm[f'spt::{rid}']   = z['z_spt']
    loaded.append(rid); label[rid] = tag

for rid in sel_ids:
    load_run(rid, 'selected')
    load_run(vanilla_of(rid), 'vanilla_MM')

joint_keys = [f'joint::{r}' for r in loaded] + ['z_scvi_abundance', 'z_scvi_spatial']
abn_keys   = [f'abn::{r}'   for r in loaded] + ['z_scvi_abundance']
spt_keys   = [f'spt::{r}'   for r in loaded] + ['z_scvi_spatial']
permod_keys = abn_keys + spt_keys
print(f'{len(loaded)} runs loaded ({sum(v=="selected" for v in label.values())} selected, '
      f'{sum(v=="vanilla_MM" for v in label.values())} vanilla)')

## Bio-preservation ×2 (label = `cell_type_annot`)

One Benchmarker on **joint** latents, one on **per-modality** latents. Both report bio-conservation (Isolated labels, KMeans NMI/ARI, Silhouette, cLISI). The joint Benchmarker's batch-correction columns are reused for the System section below.

In [ ]:
# Bio-preservation on the JOINT latent
def pretty(k):
    if '::' in k:
        scope, rid = k.split('::', 1)
        return f'{label.get(rid, "?")}|{rid.split("__", 1)[0].replace("arch-", "")}|{scope}'
    return k

def run_bm(keys):
    bm = Benchmarker(adata, batch_key=BATCH_KEY, label_key=BIO_KEY, embedding_obsm_keys=keys,
                     bio_conservation_metrics=BioConservation(),
                     batch_correction_metrics=BatchCorrection())
    bm.benchmark()
    return bm

def agg_table(bm, cols):
    res = bm.get_results(min_max_scale=False)
    res = res.loc[[i for i in res.index if i != 'Metric Type']]
    keep = [c for c in cols if c in res.columns]
    out = res[keep].astype(float).round(4)
    out.index = [pretty(i) for i in out.index]
    return out.sort_values(keep[0], ascending=False)

bm_joint = run_bm(joint_keys)
bm_joint.plot_results_table(min_max_scale=False); plt.show()
print('Bio-preservation (joint latent):')
agg_table(bm_joint, ['Bio conservation', 'Total'])

In [ ]:
# Bio-preservation on the PER-MODALITY latents (abundance z_abn + spatial z_spt)
bm_perm = run_bm(permod_keys)
bm_perm.plot_results_table(min_max_scale=False); plt.show()
print('Bio-preservation (per-modality latents):')
agg_table(bm_perm, ['Bio conservation', 'Total'])

## System integration ×1 (batch = `cell_system`, joint latent)

Batch-correction metrics (Silhouette batch, iLISI, KBET, graph connectivity, PCR) from the joint Benchmarker — how well the four experimental systems are mixed while cell type is preserved.

In [ ]:
# System integration: batch-correction columns of the joint Benchmarker
print('System integration (joint latent, batch = cell_system):')
agg_table(bm_joint, ['Batch correction', 'Total'])

## Optional: full `MultiModalVIMetrics` (GPU — reloads models)

Richer comparison (per-modality reconstruction error, NLL, latent-quality clustering) of the selected models vs their vanilla-MM counterparts. Reloads the saved modules, so it needs a GPU kernel and the modality obsm present (ct100 runs rebuild `spatial_asinh5_ct_top100` on the fly via `build_spatial_celltype_obsm`). Skip if you only need the scib tables above.

In [ ]:
from collections import defaultdict
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.metrics import MultiModalVIMetrics
from PixelGen.utils import build_spatial_celltype_obsm

cfgs  = {rid: json.loads((FINAL / rid / 'config.json').read_text()) for rid in loaded}
by_sk = defaultdict(list)
for rid in loaded:
    by_sk[cfgs[rid]['spatial_key']].append(rid)

# rebuild any ct100 spatial obsm the saved registries expect
for sk in by_sk:
    if sk not in adata.obsm:
        build_spatial_celltype_obsm(adata, celltype_key=BIO_KEY, score_key='spatial_raw',
                                    value_key='spatial_asinh5', z_thresh=2.04, top_x=100, use_abs=True)

# one MultiModalVIMetrics per spatial group (models must share modalities); use the top selected group
target_sk = cfgs[sel_ids[0]]['spatial_key']
group = by_sk[target_sk]
print('MultiModalVIMetrics on spatial group', target_sk, '->', group)

models = {}
for rid in group:
    try:
        models[pretty(f'joint::{rid}')] = MultiModalSCVI.load(str(FINAL / rid / 'model'), adata=adata)
    except Exception as e:
        print(f'  [load failed] {rid}: {e}')

if models:
    mm = MultiModalVIMetrics(adata, models, biological_key=BIO_KEY, batch_key=BATCH_KEY)
    mm.run()
    mm.plot_scib_metrics()
    mm.mean_modality_errors_barplot()